# L'or après les prix, zone par zone · *Gold after prices, zone by zone*

Notebook compagnon de l'enquête **L'or monte-t-il parce que les monnaies s'effondrent ?** — [lire l'article](https://nmlab.io/ressources/prix-de-l-or-et-effondrement-des-monnaies).
Companion notebook to the study **Is gold rising because currencies are collapsing?**.

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import numpy as np
import pandas as pd
from pandas import DataFrame

# Tableau 1 de l'article : facteurs de croissance cumulés de 1999 T1 à 2026 T1.
# La troisième colonne est publiée telle quelle — elle vient des valeurs non
# arrondies, d'où un écart possible de 0,01 avec le rapport des deux premières.
# Article table 1: cumulative growth factors, 1999 Q1 to 2026 Q1; the third column
# is published as computed on unrounded values.
FACTORS = DataFrame(
    [("US", 17.00, 1.99, 8.55), ("EA", 16.29, 1.76, 9.27),
     ("JP", 22.89, 1.15, 19.90), ("UK", 20.59, 1.95, 10.58),
     ("CH", 9.33, 1.18, 7.94), ("CA", 15.42, 1.81, 8.52),
     ("AU", 15.50, 2.16, 7.17), ("CN", 14.21, 1.64, 8.69)],
    columns=["zone", "gold", "cpi", "after"]).set_index("zone")

# Facteurs du panier, tels que publiés (calculés sur les séries non arrondies).
# La moyenne géométrique des huit facteurs ci-dessus, arrondis, donnerait ×15,92
# et ×9,57 : l'écart de 0,01 vient de ces arrondis, pas d'une autre méthode.
BASKET = {"gold": 15.93, "after": 9.57}


from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="Les prix locaux ne retirent qu'un cinquième de la hausse",
        sub="Facteur de croissance de l'or, 1999 T1 – 2026 T1 : brut, puis divisé par l'indice des prix de sa propre zone.",
        gold="Or local, brut", after="Or après les prix de la même zone",
        basket="Panier des huit zones",
        zones=dict(US="États-Unis", EA="Zone euro", JP="Japon", UK="Royaume-Uni",
                   CH="Suisse", CA="Canada", AU="Australie", CN="Chine"),
        note="Chaque zone est comparée à son propre indice de prix : un prix en yens se déflate avec les données\n"
             "japonaises. Facteurs publiés dans le tableau 1 du chapitre ; le panier est leur moyenne géométrique."),
    "en": dict(
        title="Local prices remove only a fifth of the rise",
        sub="Gold growth factor, 1999 Q1 to 2026 Q1: raw, then divided by the price index of its own zone.",
        gold="Local gold, raw", after="Gold after prices in the same zone",
        basket="Eight-zone basket",
        zones=dict(US="United States", EA="Euro area", JP="Japan", UK="United Kingdom",
                   CH="Switzerland", CA="Canada", AU="Australia", CN="China"),
        note="Each zone is compared with its own price index: a yen price is deflated with Japanese data.\n"
             "Factors as published in table 1 of the study; the basket is their geometric mean."),
}


def build_figure(table: DataFrame, basket: dict[str, float], lang: str) -> Figure:
    """Deux barres par zone — facteur brut et facteur après les prix — plus les repères du panier."""
    text = LABELS[lang]
    table = table.sort_values("after")
    factor = (lambda value: f"×{value:.2f}".replace(".", ",") if lang == "fr" else f"×{value:.2f}")

    fig = nm.figure(height_px=1180)
    ax = nm.axes(fig, left=0.118, right=0.982)
    positions = np.arange(len(table))
    for column, key, color, offset in (("gold", "gold", nm.COLORS["blue"], 0.21),
                                       ("after", "after", nm.COLORS["teal"], -0.21)):
        ax.barh(positions + offset, table[column], height=0.38, color=color,
                label=text[key], zorder=3)
        for y, value in zip(positions + offset, table[column]):
            ax.text(value + 0.28, y, factor(value), va="center", ha="left",
                    fontsize=18, color=color)

    for column, color in (("gold", nm.COLORS["blue"]), ("after", nm.COLORS["teal"])):
        ax.axvline(basket[column], color=color, lw=2, ls=(0, (5, 4)), alpha=0.65, zorder=2)
        ax.text(basket[column], len(table) - 0.44, f"{text['basket']} {factor(basket[column])}",
                ha="center", va="bottom", fontsize=17, color=color, alpha=0.95)

    ax.set_yticks(positions, [text["zones"][zone] for zone in table.index], fontsize=21)
    ax.set_xlim(0, max(table["gold"]) * 1.14)
    ax.set_ylim(-0.62, len(table) - 0.12)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"×{v:.0f}"))
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.105), ncol=2, frameon=False,
              fontsize=20.5, labelcolor="linecolor", handlelength=1.4, columnspacing=2.4)

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig


build_figure(FACTORS, BASKET, LANG)